### California Law Enforcement Killings Statistics
This notebook summarizes law enforcement incidents by place per each race,per white, and per persons of color for the state of California for the years 2020-2024

And then merges it with census place population statistics to calculate rate disparities

In [32]:
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

In [20]:
places = gpd.read_file("data_sources/cb_2024_06_place_500k.zip")

In [114]:
place_race = pd.read_csv("data/place_populations.csv", dtype={'PLACEFP':str})

In [70]:
incidents = pd.read_csv("data_sources/Mapping Police Violence.csv")

In [131]:
race_col_map = {
    'American Indian and Alaska Native':'native_incidents',
    'Asian':'asian_incidents',
    'Black':'black_incidents',
    'Hispanic':'hispanic_incidents',
    'Native Hawaiian and Pacific Islander':'pi_incidents',
    'Unknown race':'unknown_incidents',
    'White':'white_incidents',
    'poc':'poc_incidents'
}
race_cols = [
    'native', 
    'asian', 
    'black', 
    'hispanic', 
    'pi', 
    'unknown', 
    'white', 
    'poc']

In [151]:
# These two values are used in disparity calculations
# min_incidents is the minimum number of total police killing incidents per place to be considered significant enough for disparities to be calculated
# disparity_cap_value is the value to set when the white incident rates are zero and poc rates are infinity
# change these based on your own literature review or data distribution
min_incidents=5
disparity_cap_value=20.0

## Incidents Data Cleanup

We are only interested in California incidents. Since the data quantity is limited, we will use all years available (2013-2026)

In [115]:
incidents = incidents[incidents["state"] == "California"].copy().reset_index(drop=True)
len(incidents)

2124

## Join incidents to place and sum by race

In [116]:
geometry = [Point(xy) for xy in zip(incidents['longitude'], incidents['latitude'])]
incidents_gdf = gpd.GeoDataFrame(incidents, geometry=geometry, crs="EPSG:4326")

In [117]:
places = places[['PLACEFP', 'NAME', 'geometry']].to_crs(incidents_gdf.crs)

In [118]:
def aggregate_incidents(merged_df):
    """
    Aggregates incident counts and counts per race per place.
    """
    
    race_counts = merged_df.pivot_table(
        index=['PLACEFP', 'NAME'], 
        columns='race', 
        values='name',
        aggfunc='count', 
        fill_value=0
    )
    
    # Calculate total incidents per place
    race_counts['total_incidents'] = race_counts.sum(axis=1)
    
    return race_counts.reset_index()

In [119]:
places_to_incidents = gpd.sjoin(
    incidents_gdf, places, predicate='within')

In [120]:
incidents_stats = aggregate_incidents(places_to_incidents)

In [121]:
poc_races = [
    'American Indian and Alaska Native',
    'Asian',
    'Black',
    'Hispanic',
    'Native Hawaiian and Pacific Islander',
    'Unknown race'
]

In [122]:
incidents_stats['poc'] = incidents_stats[poc_races].sum(axis=1)

In [123]:
#rename columns to match census for consistency
incidents_stats = incidents_stats.rename(columns=race_col_map)

In [94]:
def calculate_race_percentages(df, race_columns):
    """
    Calculates the percentage of incidents by race for given columns.
    Appends new columns with '_pct' suffix.
    """
    for race in race_columns:
        if race in df.columns:
            pct_col_name = f"{race}_pct"
            df[pct_col_name] = (df[race] / df['total_incidents']) * 100
            
    return df

In [132]:
incidents_stats_pct = calculate_race_percentages(incidents_stats, race_col_map.values())

## Merge with census data and calculate disparity rates

In [128]:
incidents_to_place = incidents_stats.merge(place_race, on='PLACEFP')

In [152]:
def calculate_disparity_rates(df, race_cols):
    """
    Calculates how much more likely a group is to have an incident compared to White people.
    Formula: (Group Incidents / Group Population) / (White Incidents / White Population)
    """
    
    for race in race_cols:
        if race == 'white':
            continue

        disparity_col = f"{race}_likelihood_vs_white"
            
        # Incidence rates
        group_rate = df[f"{race}_incidents"] / df[f"{race}_pop"]
        white_rate = df['white_incidents'] / df['white_pop']
        
        # Likelihood ratio
        df[disparity_col] = group_rate / white_rate

        # Handle INF: White rate is 0 but Group rate > 0
        extreme_mask = (white_rate == 0) & (group_rate > 0)
        df.loc[extreme_mask, disparity_col] = disparity_cap_value

        df[disparity_col] = df[disparity_col].fillna(0.0)
        
        # If total_incidents is too small, suppress to NaN
        df.loc[df['total_incidents'] < min_incidents, disparity_col] = np.nan
        
    return df

In [153]:
incidents_disparities = calculate_disparity_rates(incidents_to_place, race_cols)

In [148]:
incidents_disparities.to_csv("data/incidents_places_disparity_rates.csv", index=False)

In [147]:
incidents_disparities.sort_values(by=["total_incidents"], ascending=False)

,PLACEFP,NAME,native_incidents,asian_incidents,black_incidents,hispanic_incidents,pi_incidents,unknown_incidents,white_incidents,total_incidents,...,hispanic_pop_pct,unknown_pop_pct,poc_pop_pct,native_likelihood_vs_white,asian_likelihood_vs_white,black_likelihood_vs_white,hispanic_likelihood_vs_white,pi_likelihood_vs_white,unknown_likelihood_vs_white,poc_likelihood_vs_white
84,15044,Compton,0,0,5,8,0,0,0,13,...,0.729444,0.016203,0.993927,NaN,NaN,inf,inf,NaN,NaN,inf
195,36546,Inglewood,0,0,8,4,0,0,0,12,...,0.485507,0.043522,0.940499,NaN,NaN,inf,inf,NaN,NaN,inf
406,73080,South Gate,0,0,0,9,0,1,0,10,...,0.951543,0.007018,0.977996,NaN,NaN,NaN,inf,NaN,inf,inf
193,36448,Indio,0,0,1,8,0,0,0,9,...,0.696487,0.017477,0.766558,NaN,NaN,inf,inf,NaN,NaN,inf
313,55618,Paramount,0,0,0,7,0,0,0,7,...,0.812738,NaN,0.958905,NaN,NaN,NaN,inf,NaN,NaN,inf
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
253,46400,Mayfair,0,0,0,1,0,0,0,1,...,0.677637,NaN,0.777359,NaN,NaN,NaN,inf,NaN,NaN,inf
103,18506,Del Mar,0,0,0,1,0,0,0,1,...,NaN,NaN,0.145017,NaN,NaN,NaN,inf,NaN,NaN,inf
256,46828,Mendota,0,0,0,1,0,0,0,1,...,0.969866,NaN,0.979725,NaN,NaN,NaN,inf,NaN,NaN,inf
260,47430,Midway City,0,1,0,0,0,0,0,1,...,0.360670,NaN,0.844856,NaN,inf,NaN,NaN,NaN,NaN,inf
